> 在线讲义：[第 21 章 · 动力系统与控制的数学：线性化、离散化、稳定性、LQR 与 Bellman 不动点](https://wikieden.github.io/math-and-eval-stats/lessons/21-dynamics-control.html) · 本书《数学与实验统计》

# 第 21 章 · 动力系统与控制的数学：线性化、离散化、稳定性、LQR 与 Bellman 不动点

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/21-dynamics-control.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 线性化 | A(θ₀) | `pendulum_A(np.pi)` | 特征值 ±3.132 |
| 第二步 离散稳定性 | \|λ\| | `zoh(pendulum_A(np.pi), B, 0.1)` | 1.368、0.731 |
| 第三步 ZOH | 矩阵指数 | `zoh(a, b, 0.1)` | A_d = [[1, 0.1], [0, 1]]，B_d = [0.005, 0.1] |
| 第四步 LQR | Riccati | `lqr_gain(ad, bd, I, [[1]])` | K = [0.917, 1.636] |
| 第五步 Bellman | 不动点 | `bellman_policy_value()` | [6.7273, 4.9091]，149 次 |

例子：单摆在下垂点和倒立点线性化；双积分器（推一个小车）用 ZOH 离散化、求 LQR；一个 2 状态的 MDP 当 Bellman 不动点。
里程碑 ME-M17 的倒立摆 LQR 闭环仿真和 4×4 网格值迭代留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.linalg import expm, solve_discrete_are
G_L = 9.81  # g/l，摆长 1 米
DT = 0.1

**函数 `pendulum_A`**：单摆 θ'' = −(g/l) sin θ，状态 [θ, θ']，在 θ0 处线性化：A = [[0, 1], [−(g/l)cos θ0, 0]]。

In [ ]:
def pendulum_A(theta0: float) -> np.ndarray:
    """单摆 θ'' = −(g/l) sin θ，状态 [θ, θ']，在 θ0 处线性化：A = [[0, 1], [−(g/l)cos θ0, 0]]。"""
    return np.array([[0.0, 1.0], [-G_L * np.cos(theta0), 0.0]])

**函数 `zoh`**：零阶保持离散化：A_d = e^{A dt}，B_d = ∫₀^dt e^{A s} ds · B（用一个扩展矩阵一次算出）。

In [ ]:
def zoh(a: np.ndarray, b: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray]:
    """零阶保持离散化：A_d = e^{A dt}，B_d = ∫₀^dt e^{A s} ds · B（用一个扩展矩阵一次算出）。"""
    n, m = a.shape[0], b.shape[1]
    big = np.zeros((n + m, n + m))
    big[:n, :n], big[:n, n:] = a, b
    e = expm(big * dt)
    return e[:n, :n], e[:n, n:]

**函数 `lqr_gain`**：离散 LQR：解代数 Riccati 方程得 P，K = (R + BᵀPB)⁻¹BᵀPA，控制 u = −Kx。

In [ ]:
def lqr_gain(ad, bd, q, r) -> tuple[np.ndarray, np.ndarray]:
    """离散 LQR：解代数 Riccati 方程得 P，K = (R + BᵀPB)⁻¹BᵀPA，控制 u = −Kx。"""
    p = solve_discrete_are(ad, bd, q, r)
    k = np.linalg.solve(r + bd.T @ p @ bd, bd.T @ p @ ad)
    return k, p

**函数 `bellman_policy_value`**：2 状态 MDP（编的）：固定策略下转移 P = [[0.8, 0.2], [0.3, 0.7]]，奖励 r = [1, 0]。

In [ ]:
def bellman_policy_value(gamma: float = 0.9) -> tuple[np.ndarray, np.ndarray, int]:
    """2 状态 MDP（编的）：固定策略下转移 P = [[0.8, 0.2], [0.3, 0.7]]，奖励 r = [1, 0]。
    精确解 V = (I − γP)⁻¹r，和反复套 V ← r + γPV 的结果对比。返回 (精确 V, 迭代 V, 迭代次数)。"""
    p = np.array([[0.8, 0.2], [0.3, 0.7]])
    r = np.array([1.0, 0.0])
    exact = np.linalg.solve(np.eye(2) - gamma * p, r)
    v, k = np.zeros(2), 0
    while np.abs(v - exact).max() > 1e-6:
        v, k = r + gamma * p @ v, k + 1
    return exact, v, k

**函数 `fmt_eigs`**：把特征值打印成 "±3.132i" 或 "3.132, −3.132" 这样好读的样子。

In [ ]:
def fmt_eigs(a: np.ndarray) -> str:
    """把特征值打印成 "±3.132i" 或 "3.132, −3.132" 这样好读的样子。"""
    out = []
    for z in np.linalg.eigvals(a):
        if abs(z.imag) < 1e-12:
            out.append(f"{z.real:.3f}")
        else:
            out.append(f"{z.real:.3f}{z.imag:+.3f}i" if abs(z.real) > 1e-12 else f"{z.imag:+.3f}i")
    return "、".join(out)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 线性化：在平衡点附近，非线性系统看起来像 ẋ = Ax")
for name, th in (("下垂点 θ = 0", 0.0), ("倒立点 θ = π", np.pi)):
    a = pendulum_A(th)
    print(f"  {name}：A = {np.round(a, 2).tolist()}，特征值 {fmt_eigs(a)}")

In [ ]:
print("\n第二步 稳定性看特征值：连续时间看实部 < 0，离散时间看模 < 1")
ad, _ = zoh(pendulum_A(np.pi), np.array([[0.0], [1.0]]), DT)
print(f"  倒立点离散化（dt = 0.1）后特征值的模 {np.round(np.abs(np.linalg.eigvals(ad)), 4).tolist()}：有一个 > 1，不稳定")

In [ ]:
print("\n第三步 ZOH 离散化：双积分器 ẍ = u")
a = np.array([[0.0, 1.0], [0.0, 0.0]])
b = np.array([[0.0], [1.0]])
ad, bd = zoh(a, b, DT)
print(f"  A_d = {np.round(ad, 4).tolist()}，B_d = {np.round(bd, 4).ravel().tolist()}（= [dt²/2, dt]）")

In [ ]:
print("\n第四步 LQR：Q = I、R = 1 时的反馈增益")
k, p = lqr_gain(ad, bd, np.eye(2), np.array([[1.0]]))
cl = ad - bd @ k
print(f"  K = {np.round(k, 4).ravel().tolist()}；闭环特征值的模 {np.round(np.abs(np.linalg.eigvals(cl)), 4).tolist()}（都 < 1，稳了）")
k2, _ = lqr_gain(ad, bd, np.eye(2), np.array([[0.01]]))
print(f"  R 从 1 降到 0.01（控制便宜）：K = {np.round(k2, 4).ravel().tolist()}，更激进")

In [ ]:
print("\n第五步 Bellman 方程是不动点方程")
exact, v, n = bellman_policy_value()
print(f"  精确解 V = {np.round(exact, 4).tolist()}；反复套 V ← r + γPV，{n} 次误差 < 1e-6：{np.round(v, 4).tolist()}")

## 3. 代码里才看得到的

1. **扩展矩阵的技巧**：直接算 ∫e^{As}ds 需要 A 可逆；把 A、B 拼成一个大矩阵再取指数，对奇异的 A（比如双积分器）也成立。
2. **`solve_discrete_are` 的参数顺序**：(A, B, Q, R)，和连续时间的 `solve_continuous_are` 同样的顺序；混用离散 / 连续版本是常见错误。
3. **特征值打印成复数**：`np.linalg.eigvals` 对实矩阵也可能返回复数（共轭对），比较稳定性时要取实部或模。

## 4. 自己改一改

1. 把 LQR 的 Q 改成 diag(10, 1)（更在意位置误差），K 怎么变？
2. 在 `bellman_policy_value` 里把 γ 改成 0.99，迭代次数变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 门口街道看着平，平衡点处线性化，特征值定命运；
> 连续看实部、离散看模，倒立摆有一个越界；
> 零阶保持积一周期，A_d 是矩阵指数；
> LQR 用二次代价换线性反馈，R 小就激进；Bellman 是不动点，xᵀPx 就是值函数。